In [6]:
import os
import glob
import polars as pl

In [7]:
def remove_columns_from_files(input_dir: str, cols_to_drop: list, use_output_dir: bool = False):
    output_dir = os.path.join(input_dir, "cleaned")
    
    if use_output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    extensions = ["*.csv", "*.parquet"]
    print(f"Scanning directory for dropping columns: {input_dir}...")

    for ext in extensions:
        pattern = os.path.join(input_dir, ext)
        files = glob.glob(pattern)
        
        for filepath in files:
            filename = os.path.basename(filepath)
            
            try:
                if filename.lower().endswith(".parquet"):
                    df = pl.read_parquet(filepath)
                else:
                    df = pl.read_csv(filepath)
                
                # Check which columns actually exist in the dataframe
                existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]
                
                # If none of the columns exist, skip processing this file
                if not existing_cols_to_drop:
                    print(f"[SKIPPED] No columns to drop for: {filename}")
                    continue
                
                df_clean = df.drop(existing_cols_to_drop)
                
                out_path = os.path.join(output_dir, filename) if use_output_dir else filepath
                tmp_path = out_path + ".tmp"
                
                # Write to a temporary file first to avoid OS Error 1224 (Windows file lock)
                if filename.lower().endswith(".parquet"):
                    df_clean.write_parquet(tmp_path)
                else:
                    df_clean.write_csv(tmp_path)
                    
                # Replace the original file with the temporary file safely
                os.replace(tmp_path, out_path)
                    
                print(f"[SUCCESS] Dropped columns {existing_cols_to_drop} for: {filename}")

            except Exception as e:
                print(f"[ERROR] Failed to process {filename}: {e}")

def add_columns_to_files(input_dir: str, new_columns: list, use_output_dir: bool = False, insert_before: str = None, insert_after: str = None):
    output_dir = os.path.join(input_dir, "cleaned")
    
    if use_output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    extensions = ["*.csv", "*.parquet"]
    print(f"Scanning directory for adding/casting columns: {input_dir}...")

    for ext in extensions:
        pattern = os.path.join(input_dir, ext)
        files = glob.glob(pattern)
        
        for filepath in files:
            filename = os.path.basename(filepath)
            
            try:
                if filename.lower().endswith(".parquet"):
                    df = pl.read_parquet(filepath)
                else:
                    df = pl.read_csv(filepath)
                
                exprs = []
                for col in new_columns:
                    if col in df.columns:
                        exprs.append(pl.col(col).cast(pl.Int64))
                    else:
                        exprs.append(pl.lit(None, dtype=pl.Int64).alias(col))
                
                df_clean = df.with_columns(exprs)
                
                if insert_before or insert_after:
                    current_cols = df_clean.columns
                    base_cols = [c for c in current_cols if c not in new_columns]
                    ref_col = insert_before if insert_before else insert_after
                    
                    if ref_col in base_cols:
                        idx = base_cols.index(ref_col)
                        if insert_after:
                            idx += 1
                        
                        final_cols = base_cols[:idx] + new_columns + base_cols[idx:]
                        df_clean = df_clean.select(final_cols)
                
                out_path = os.path.join(output_dir, filename) if use_output_dir else filepath
                tmp_path = out_path + ".tmp"
                
                if filename.lower().endswith(".parquet"):
                    df_clean.write_parquet(tmp_path)
                else:
                    df_clean.write_csv(tmp_path)
                    
                os.replace(tmp_path, out_path)
                    
                print(f"[SUCCESS] Processed and standardized columns {new_columns} for: {filename}")

            except Exception as e:
                print(f"[ERROR] Failed to process {filename}: {e}")
                
if __name__ == "__main__":
    remove_fol = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\STORAGE_OUTPUT_RTA"
    add_fol = r"C:\Users\ADMIN\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\RAW\OUTPUT_PERFORMANCE\OUTPUT_PERFORMANCE_COMBINE"

    columns_to_remove = ["total_time_pull_out"]
    columns_to_add = ["Duplicate_Flag"]

    remove_columns_from_files(remove_fol, columns_to_remove, use_output_dir=False)
    # add_columns_to_files(add_fol, columns_to_add, use_output_dir=False, insert_before="Quartile Flag")

Scanning directory for dropping columns: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\STORAGE_OUTPUT_RTA...
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-01-01.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-01-08.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-01-15.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-01-22.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-01-29.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-02-05.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-02-12.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-02-19.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-02-26.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-03-04.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-03-11.csv
[SUCCESS] Dropped columns ['total_time_pull_out'] for: 2024-03-18.csv
[SUCCESS